In [0]:
# Parameters passed from the job
# These should match the target's catalog and schema variables:
#   dev:  catalog='datasets', schema=<username>
#   main: catalog='datasets', schema='stage'
#   prod: catalog='datasets', schema='prod'
dbutils.widgets.text('catalog', 'datasets')
dbutils.widgets.text('schema', 'default')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')

print(f"Using catalog: {catalog}, schema: {schema}")

In [0]:
import requests
import json
import time
import random
from pathlib import Path
from datetime import datetime


# DataUSA API Configuration
base_url = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
)

params = {
    "cube": "acs_yg_total_population_1",
    "drilldowns": "Year,Nation",
    "locale": "en",
    "measures": "Population",
    # Number of records returned per API call.
    "limit": 5000,
    # Pagination starts at record 0.
    # The loop increases this by "limit" after each page.
    "offset": 0
}

# Output Configuration
# Add date-based folder structure: YYYY/MM/DD
today = datetime.now()
date_path = today.strftime("%Y/%m/%d")
output_dir = Path(f"/Volumes/{catalog}/{schema}/quest_data/datausa_data/{date_path}")
output_dir.mkdir(parents=True, exist_ok=True)

# API Timing Configuration
    # sleep_seconds = 0.2
    #   ~5 requests/sec
    #   Good for quick testing
    #   More aggressive
    #
    # sleep_seconds = 1
    #   1 request/sec
    #   Recommended default for production
    #
    # sleep_seconds = 5
    #   Large historical extracts
    #   Running scheduled batch jobs
sleep_seconds = 1


# Retry Configuration
max_retries = 5


def get_api_response(url, params):
    """
    Calls API with retry handling.

    Handles:
        200 - Success
        429 - Rate limited
        5xx - Temporary server issue

    Uses exponential backoff:

        retry 1 -> wait ~2 seconds
        retry 2 -> wait ~4 seconds
        retry 3 -> wait ~8 seconds

    This prevents hammering the API during outages.
    """

    retry_count = 0

    while retry_count < max_retries:

        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        # Successful response
        if response.status_code == 200:
            return response.json()

        # --------------------------------------------
        # Rate limited
        #
        # Example:
        # HTTP 429 Too Many Requests
        #
        # Wait longer each retry.
        # --------------------------------------------
        if response.status_code == 429:

            wait_time = (
                (2 ** retry_count)
                + random.random()
            )

            print(
                f"Rate limited. "
                f"Waiting {wait_time:.2f} seconds..."
            )

            time.sleep(wait_time)

            retry_count += 1
            continue

        # --------------------------------------------
        # Temporary server failures
        #
        # Examples:
        # 500 Internal Server Error
        # 502 Bad Gateway
        # 503 Service Unavailable
        # --------------------------------------------
        if response.status_code >= 500:

            wait_time = (
                (2 ** retry_count)
                + random.random()
            )

            print(
                f"Server error {response.status_code}. "
                f"Retrying in {wait_time:.2f} seconds..."
            )

            time.sleep(wait_time)

            retry_count += 1
            continue

        # --------------------------------------------
        # Other errors
        #
        # Example:
        # Invalid parameters
        #
        # Do not retry because it will fail again.
        # --------------------------------------------
        response.raise_for_status()

    raise Exception(
        "Maximum API retries exceeded"
    )

# Main Extraction Loop
page_number = 0

while True:
    offset = params["offset"]

    print(
        f"Downloading page {page_number} "
        f"(offset={offset})"
    )

    # Call API
    result = get_api_response(
        base_url,
        params
    )

    # DataUSA places records under "data"
    records = result.get(
        "data",
        []
    )

    # No records means extraction is complete
    if not records:
        print(
            "No additional records returned."
        )
        break

    # ========================================================
    # Save COMPLETE API response
    #
    # Do NOT save only "data".
    #
    # Keeping the entire response preserves:
    #
    #   - data
    #   - metadata
    #   - columns
    #   - source information
    #
    # This is the raw Bronze layer.
    # ========================================================

    output_file = (
        output_dir
        / f"page_{page_number:04d}_offset_{offset}.json"
    )

    with output_file.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"Saved {output_file} "
        f"({len(records)} records)"
    )

    if len(records) < params["limit"]:

        print(
            "Final page reached."
        )

        break


    params["offset"] += params["limit"]

    page_number += 1

    # Delay between successful calls
    time.sleep(
        sleep_seconds
    )

print(
    f"Completed extraction. "
    f"Saved {page_number + 1} pages."
)